# 02 · E1: línea base con un encoder clínico ajustado

### Qué es este experimento

La literatura internacional coincide en que, para **reconocer entidades clínicas (NER)**, la estrategia que mejor funciona es tomar un modelo tipo BERT ya preentrenado con texto clínico y **ajustarlo** (*fine-tuning*) con ejemplos anotados de la tarea. Esto es E1. Es la vara con la que mediremos todo lo demás (los LLM en E2, la destilación en E4).

**Analogía:** un médico recién egresado (el modelo preentrenado) ya "sabe medicina". El ajuste fino es su beca de especialidad: con casos reales anotados por expertos, aprende a hacer muy bien una tarea concreta, en este caso subrayar enfermedades, fármacos y abreviaturas en derivaciones chilenas.

### Qué comparamos

| Modelo | Preentrenado con | Representa |
|---|---|---|
| `PlanTL-GOB-ES/bsc-bio-ehr-es` | Textos biomédicos y clínicos de España (BSC) | El estado del arte en español "de manual" |
| `plncmm/roberta-clinical-wl-es` | Lo anterior + derivaciones chilenas de listas de espera (PLN@CMM, U. de Chile) | El mismo modelo con "pasantía" en texto chileno |

Si el modelo chileno gana, sobre todo en las entidades abreviadas, es evidencia de que **adaptar el modelo al registro local importa**.

### Conexión con el curso

- **Unidad 2:** el ciclo de entrenamiento está escrito a mano (lotes, pérdida, retropropagación, optimizador AdamW, *early stopping*).
- **Unidad 3:** *transfer learning*, porque reutilizamos un modelo preentrenado.
- **Unidad 4:** el modelo es un Transformer de tipo *encoder*, y lo evaluamos con F1 estricto y laxo e intervalos por *bootstrap*.
- **Unidad 7:** los encoders se preentrenaron de forma autosupervisada (lenguaje enmascarado), y comparamos el efecto de la adaptación de dominio.

In [ ]:
from pathlib import Path
import sys, json, time
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

import yaml
import pandas as pd
import matplotlib.pyplot as plt
from datos.cwlc import cargar_split
from modelos.e1_encoder import Config, Codificador, etiquetas_palabra, entrenar, predecir, evaluar, guardar_resultados, dispositivo
from evaluacion import metricas

cfg_global = yaml.safe_load(open(RAIZ / "configs" / "experimentos.yaml"))
carpeta = RAIZ / "data" / "raw" / "cwlc"
train, dev, test = (cargar_split(carpeta, s) for s in ["train", "dev", "test"])
print(f"train={len(train)}  dev={len(dev)}  test={len(test)}  | dispositivo: {dispositivo()}")

## 1. Por qué un modelo por tipo de entidad

En el CWLC las entidades se **anidan**: en "pcte con DM2", la palabra "DM2" es a la vez una *Enfermedad* y una *Abreviatura*. Un etiquetador normal le pone **una** etiqueta a cada palabra, así que no podría marcar ambas.

La solución que usaron los autores del corpus (Báez et al., 2022; Rojas et al., 2022) es entrenar **un etiquetador independiente por tipo**. Cada uno solo aprende a responder "¿esta palabra es parte de una Enfermedad?" (o de un Fármaco, o de una Abreviatura). Hacemos lo mismo para poder comparar.

Las etiquetas usan el esquema **BIO**: `B-` empieza una entidad, `I-` la continúa y `O` es todo lo demás.

In [ ]:
o = next(x for x in train if any(e[3] == "Disease" for e in x.entidades) and any(e[3] == "Abbreviation" for e in x.entidades))
pd.DataFrame({"palabra": o.tokens,
              "Disease": etiquetas_palabra(o, "Disease"),
              "Abbreviation": etiquetas_palabra(o, "Abbreviation")}).T

## 2. Palabras y subpalabras

El modelo no ve palabras: ve **subpalabras** (*tokens*). Las palabras poco frecuentes, como muchas abreviaturas chilenas, se parten en pedazos. Solo la **primera subpalabra** de cada palabra lleva la etiqueta; las demás reciben `-100`, el valor que PyTorch usa para decir "no evalúes esta posición".

Fíjense en cómo cada modelo parte "pcte c/ DM2 desc". Cuantos más pedazos, más le cuesta al modelo reconocer el concepto.

In [ ]:
for nombre in cfg_global["encoders"].values():
    cod = Codificador(nombre, "Disease", 512)
    enc = cod.codificar(["pcte", "c/", "DM2", "desc"], ["O", "O", "B-Disease", "I-Disease"])
    subp = cod.tok.convert_ids_to_tokens(enc["input_ids"])
    print(f"{nombre}\n  subpalabras: {subp}\n  etiquetas:   {enc['labels']}\n")

## 3. Configuración del experimento

- **`MODO = "rapido"`** entrena con 800 oraciones y 2 épocas, para comprobar que todo funciona (unos minutos).
- **`MODO = "completo"`** usa las 8.014 oraciones de entrenamiento y hasta 5 épocas con *early stopping*. Esta es la versión que se reporta.

Hiperparámetros, desde `configs/experimentos.yaml`:

- **Tasa de aprendizaje 3×10⁻⁵ con calentamiento lineal:** los pesos preentrenados se mueven poco, para no "olvidar" lo aprendido.
- **Lote de 16 oraciones.**
- **Hasta 5 épocas:** el entrenamiento se detiene si el F1 en *dev* no mejora en 2 épocas seguidas.

El *test* se usa **una sola vez**, al final.

In [ ]:
MODO = "rapido"          # "rapido" para probar | "completo" para reportar
MODELOS = cfg_global["encoders"]                       # {"espanol_clinico": ..., "chileno_clinico": ..., "general": ...}
MODELOS = {k: v for k, v in MODELOS.items() if k in ["espanol_clinico", "chileno_clinico"]}
TIPOS = cfg_global["entidades"]                        # ["Disease", "Medication", "Abbreviation"]
ent = cfg_global["entrenamiento_encoder"]

def config_para(modelo, tipo):
    if MODO == "rapido":
        return Config(modelo=modelo, tipo=tipo, epocas=2, lr=5e-5, batch=ent["batch"], max_train=800, semilla=cfg_global["semilla"])
    return Config(modelo=modelo, tipo=tipo, epocas=ent["epocas"], lr=ent["lr"], batch=ent["batch"], semilla=cfg_global["semilla"])

SALIDA = RAIZ / "results" / f"e1_{MODO}"
print(MODO, "|", list(MODELOS), "|", TIPOS, "->", SALIDA)

## 4. Entrenar y evaluar

Para cada modelo y cada tipo de entidad, la celda:

1. Ajusta el modelo con *train* y elige la mejor época según el F1 en *dev*.
2. Predice sobre *test*.
3. Calcula las métricas: precisión, *recall* y F1 **estricto** (inicio, fin y tipo exactos) y **laxo** (basta con superponerse), con intervalo de confianza del 95% por *bootstrap*. Además, el *recall* en entidades **abreviadas frente a no abreviadas**.
4. Guarda los resultados en `results/` (los JSON de métricas sí se suben a git; las predicciones no).

En el Mac M5 con MPS, el modo completo toma del orden de minutos por época y por modelo. La primera corrida registra los tiempos reales en el historial.

In [ ]:
predicciones = {}
for clave, nombre in MODELOS.items():
    for tipo in TIPOS:
        print(f"\n=== {clave} · {tipo} ===")
        cfg = config_para(nombre, tipo)
        t0 = time.time()
        modelo, cod, historial = entrenar(cfg, train, dev)
        pred = predecir(modelo, cod, test, tipo, disp=dispositivo())
        gold = [e for o in test for e in o.entidades if e[3] == tipo]
        res = evaluar(gold, pred, test, tipo)
        res["minutos_total"] = round((time.time() - t0) / 60, 1)
        guardar_resultados(SALIDA / f"{clave}_{tipo}.json", cfg, historial, res)
        predicciones[(clave, tipo)] = pred
        print(f"  F1 estricto test = {res['estricto']['f1']:.3f}  IC95 {res['estricto']['ic95'][0]:.3f}–{res['estricto']['ic95'][1]:.3f}")
        del modelo

## 5. Resultados

La tabla junta las métricas de cada combinación de modelo y tipo de entidad. Cómo leerla:

- **F1 estricto:** la métrica principal. Exige marcar exactamente las mismas palabras que el médico.
- **F1 laxo:** perdona errores de límites (por ejemplo, marcar "diabetes mellitus" en vez de "diabetes mellitus tipo 2"). La diferencia entre laxo y estricto mide cuántos errores son solo de borde.
- **Recall abreviadas / no abreviadas:** de las entidades que el médico marcó, qué proporción encontró el modelo en cada grupo (hipótesis H3).

In [ ]:
filas = []
for f in sorted(SALIDA.glob("*.json")):
    d = json.loads(f.read_text())
    r, c = d["resultados"], d["config"]
    filas.append({"modelo": f.stem.rsplit("_", 1)[0], "tipo": c["tipo"],
                  "P": r["estricto"]["precision"], "R": r["estricto"]["recall"], "F1 estricto": r["estricto"]["f1"],
                  "IC95": f"{r['estricto']['ic95'][0]:.3f}–{r['estricto']['ic95'][1]:.3f}", "F1 laxo": r["laxo"]["f1"],
                  "recall abrev.": r.get("abreviadas", {}).get("recall_estricto"),
                  "n abrev.": r.get("abreviadas", {}).get("n_gold"),
                  "recall no abrev.": r.get("no_abreviadas", {}).get("recall_estricto"),
                  "épocas": len(d["historial"]), "min": r.get("minutos_total")})
resultados = pd.DataFrame(filas).round(3)
resultados

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
piv = resultados.pivot(index="tipo", columns="modelo", values="F1 estricto")
piv.plot.bar(ax=ax, rot=0)
ax.set_ylabel("F1 estricto (test)"); ax.set_ylim(0, 1); ax.set_title("E1 · Encoder ajustado por tipo de entidad")
ax.legend(title="", loc="lower right"); plt.tight_layout(); plt.show()

## 6. ¿El modelo chileno es mejor que el español?

Comparamos ambos modelos sobre **las mismas oraciones** con *bootstrap pareado*: se remuestrean las oraciones del test 1.000 veces y se calcula la diferencia de F1 en cada remuestreo. Si el intervalo del 95% de la diferencia **no incluye el 0**, la diferencia es significativa.

In [ ]:
if {"espanol_clinico", "chileno_clinico"} <= {k for k, _ in predicciones}:
    for tipo in TIPOS:
        gold = [e for o in test for e in o.entidades if e[3] == tipo]
        a = metricas.por_documento(gold, predicciones[("chileno_clinico", tipo)], "estricto")
        b = metricas.por_documento(gold, predicciones[("espanol_clinico", tipo)], "estricto")
        dif, lo, hi = metricas.bootstrap_pareado(a, b)
        veredicto = "significativa" if lo > 0 or hi < 0 else "no significativa"
        print(f"{tipo:<13} F1(chileno) − F1(español) = {dif:+.3f}  IC95 [{lo:+.3f}, {hi:+.3f}]  -> {veredicto}")

## 7. Contexto: ¿cómo se compara con la literatura?

Referencias publicadas con el CWLC (no son directamente comparables, porque usan otra arquitectura o todas las entidades juntas):

| Trabajo | Corpus | Resultado |
|---|---|---|
| Báez et al. 2020 | Piloto, 900 derivaciones | F1 por tipo: Abbreviation 0,92 · Disease 0,77 · Medication 0,75 |
| Báez et al. 2022 | 5.000 derivaciones (estos mismos *splits*) | F1 global ≈ 0,80 (citado por Rojas 2022) |
| Rojas et al. 2022 | Otra partición, 7 tipos | F1 global 0,83 (mejor publicado) |

Si nuestros F1 por tipo quedan en ese rango, la línea base es sólida.

## 8. Conclusiones (completar después de la corrida completa)

- F1 estricto por tipo, y cuál es más difícil:
- ¿El modelo chileno supera al español? ¿En qué tipos?
- Recall en abreviadas frente a no abreviadas (H3):
- Tiempo de entrenamiento en el Mac:

**Siguiente notebook:** `03_e2_llm_prompting.ipynb`, donde los LLM locales intentan la misma tarea solo con instrucciones.